# 07 FM — Anomaly Detection with Foundation Models & Embeddings

This notebook is the **Foundation Model (FM) companion** to `07_anomaly_detection_production.ipynb`,
which implements classical Z-score and Isolation Forest baselines.  Here we apply three
FM-based approaches and compare them against those baselines.

## Why go beyond classical methods?

| Classical | FM-based |
|---|---|
| Z-score flags a point if its YoY change is extreme *within that series* | Chronos flags a point if the model — trained on millions of time-series — genuinely "didn't expect" it |
| Isolation Forest uses hand-crafted statistical features | Autoencoder learns its own compact representation of production dynamics |
| No semantic understanding of the data | Sentence-transformer embeddings encode the *meaning* of each data record; outliers in embedding space differ semantically, not just numerically |

## Methods covered

1. **Classical baselines** — Z-score (|z| > 2.5) and Isolation Forest, reproduced concisely for comparison.
2. **Chronos** (Amazon) — time-series FM.  We use an expanding-window forecast: predict each year from all prior years in the series.  Large forecast error → anomaly.
3. **Sentence-transformer embeddings** (BAAI/bge-large-en-v1.5) — encode each (country, commodity, year, quantity, YoY) record as a semantic vector; flag points far from their commodity's centroid.
4. **PyTorch autoencoder** — learns a low-dimensional representation of series-level statistical features; high reconstruction error → the series is hard to explain → anomalous.

In [ ]:

import warnings; warnings.filterwarnings('ignore')
import pathlib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import IsolationForest
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import torch
from sentence_transformers import SentenceTransformer

DATA_DIR = pathlib.Path("../data/bgs_data")
CSV_PATH = DATA_DIR / "bgs_critical_minerals_production.csv"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Torch device : {DEVICE}")
print(f"Data path    : {CSV_PATH.resolve()}")

## Data Loading & Preparation

Steps mirror the classical notebook exactly so that results are directly comparable:

1. Load raw BGS CSV.
2. Keep `statistic_type == 'Production'` rows.
3. Coerce numerics, drop rows missing key fields.
4. Aggregate duplicates (sum) within (commodity, country, year).
5. Sort chronologically and compute YoY changes per (commodity, country) series.

In [ ]:
raw = pd.read_csv(CSV_PATH, low_memory=False)

# Keep production rows only
df = raw[raw["statistic_type"] == "Production"].copy()

# Coerce numerics
df["year"]     = pd.to_numeric(df["year"],     errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df = df.dropna(subset=["year", "quantity", "commodity", "country"])
df["year"] = df["year"].astype(int)

# Aggregate duplicates
df = (
    df.groupby(["commodity", "country", "year"], as_index=False)["quantity"]
    .sum()
)
df = df.sort_values(["commodity", "country", "year"]).reset_index(drop=True)

print(f"Production records  : {len(df):,}")
print(f"Commodities         : {df['commodity'].nunique()}")
print(f"Countries           : {df['country'].nunique()}")
print(f"Year range          : {df['year'].min()} – {df['year'].max()}")

In [ ]:
# Year-over-year change (same formula as classical notebook)
grp = df.groupby(["commodity", "country"])
df["prev_quantity"] = grp["quantity"].shift(1)
df["yoy_change"]    = (df["quantity"] - df["prev_quantity"]) / df["prev_quantity"]
df["yoy_change"]    = df["yoy_change"].replace([np.inf, -np.inf], np.nan)

df_yoy = df.dropna(subset=["prev_quantity", "yoy_change"]).copy()

print(f"Rows with valid YoY : {len(df_yoy):,}")
print(df_yoy["yoy_change"].describe().round(3).to_string())

## 1. Classical Baselines (for comparison)

We reproduce the two classical methods from `07_anomaly_detection_production.ipynb` in
condensed form so that later comparison cells have consistent variable names.

* **Z-score** flags point anomalies where `|z| > 2.5` within each (commodity, country) YoY series.
* **Isolation Forest** flags entire series that are outliers in a 5-feature statistical space.

In [ ]:
# ── Z-score baseline ────────────────────────────────────────────────
ZSCORE_THRESHOLD = 2.5

def group_zscore(series: pd.Series) -> pd.Series:
    if len(series) < 3:
        return pd.Series(np.nan, index=series.index)
    mu, sigma = series.mean(), series.std(ddof=1)
    if sigma == 0 or np.isnan(sigma):
        return pd.Series(0.0, index=series.index)
    return (series - mu) / sigma

df_yoy["yoy_zscore"]        = df_yoy.groupby(["commodity", "country"])["yoy_change"].transform(group_zscore)
df_yoy["is_anomaly_zscore"] = df_yoy["yoy_zscore"].abs() > ZSCORE_THRESHOLD

n_zscore = df_yoy["is_anomaly_zscore"].sum()
print(f"Z-score point anomalies (|z| > {ZSCORE_THRESHOLD}) : {n_zscore:,}")

# ── Isolation Forest baseline ───────────────────────────────────────────────
MIN_YEARS_IF = 5
feat_rows = []
for (commodity, country), g in df_yoy.groupby(["commodity", "country"]):
    yoy = g["yoy_change"].dropna()
    qty = g["quantity"].dropna()
    if len(yoy) < MIN_YEARS_IF:
        continue
    qty_mean = qty.mean()
    feat_rows.append({
        "commodity"  : commodity,
        "country"    : country,
        "mean_yoy"   : yoy.mean(),
        "std_yoy"    : yoy.std(ddof=1),
        "max_drop"   : yoy.min(),
        "max_spike"  : yoy.max(),
        "volatility" : qty.std(ddof=1) / qty_mean if qty_mean != 0 else np.nan,
    })

classical_feat_df  = pd.DataFrame(feat_rows).dropna()
iso_feature_cols   = ["mean_yoy", "std_yoy", "max_drop", "max_spike", "volatility"]
X_if = classical_feat_df[iso_feature_cols].values
iso  = IsolationForest(contamination=0.1, random_state=42)
classical_feat_df["if_label"]     = iso.fit_predict(X_if)
classical_feat_df["if_score"]     = iso.decision_function(X_if)
classical_feat_df["is_anomaly_if"] = classical_feat_df["if_label"] == -1

n_if = classical_feat_df["is_anomaly_if"].sum()
print(f"Isolation Forest anomalous series          : {n_if:,}")

## 2. Chronos-Based Anomaly Detection

**Amazon Chronos** is a pre-trained time-series foundation model (based on T5) that has been
fine-tuned on a large corpus of real and synthetic time-series data.  We exploit it for
anomaly detection via the *reconstruction-error* principle:

> If the model — trained on diverse global time-series — cannot explain an observed value
> given all prior years, the value is likely anomalous.

**Algorithm (expanding-window):**

```
for each (commodity, country) series with >= 10 years:
    for t = 5 to T:
        context  = quantities[0 : t]
        forecast = Chronos.predict(context, prediction_length=1, num_samples=20)
        z_score  = |actual[t] - median(forecast)| / (std(forecast) + eps)
        if z_score > 3.0: flag as anomaly
```

> **Performance note:** Running Chronos over the full dataset can take tens of minutes.
> Adjust `COMMODITIES_TO_CHECK` below to a smaller subset for faster execution during
> development.  Set it to `None` to run on all commodities.

In [ ]:
from chronos import ChronosPipeline

# ── Scope control — set to None to run all commodities ─────────────────────
COMMODITIES_TO_CHECK = [
    "lithium minerals",
    "cobalt",
    "rare earth elements",
    "nickel",
    "manganese",
]
# ─────────────────────────────────────────────────────────────────────────────

CHRONOS_Z_THRESHOLD = 3.0
MIN_YEARS_CHRONOS   = 10
MIN_CONTEXT         = 5

print("Loading Chronos (amazon/chronos-t5-small) …")
chronos_pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=DEVICE,
    torch_dtype=torch.float32,
)
print("Model loaded.")

# Filter to requested commodities
if COMMODITIES_TO_CHECK is not None:
    pattern = "|".join(COMMODITIES_TO_CHECK)
    df_chronos_scope = df[
        df["commodity"].str.contains(pattern, case=False, na=False)
    ]
else:
    df_chronos_scope = df

print(f"Commodities in scope : {df_chronos_scope['commodity'].nunique()}")
print(f"Series in scope      : {df_chronos_scope.groupby(['commodity','country']).ngroups}")

chronos_anomalies = []
series_groups     = df_chronos_scope.groupby(["commodity", "country"])

for (commodity, country), group in series_groups:
    group = group.sort_values("year")
    if len(group) < MIN_YEARS_CHRONOS:
        continue

    quantities = group["quantity"].values.astype(float)
    years      = group["year"].values.astype(int)

    for i in range(MIN_CONTEXT, len(quantities)):
        context  = torch.tensor(quantities[:i], dtype=torch.float32)
        forecast = chronos_pipeline.predict(context, prediction_length=1, num_samples=20)
        # forecast shape: (num_series=1, num_samples=20, prediction_length=1)
        samples  = forecast[0].numpy().flatten()

        predicted_median = np.median(samples)
        predicted_std    = np.std(samples) + 1e-6
        actual           = quantities[i]
        z_score          = abs(actual - predicted_median) / predicted_std

        if z_score > CHRONOS_Z_THRESHOLD:
            chronos_anomalies.append({
                "commodity"        : commodity,
                "country"          : country,
                "year"             : int(years[i]),
                "actual"           : actual,
                "predicted_median" : predicted_median,
                "predicted_std"    : predicted_std,
                "z_score"          : z_score,
            })

chronos_anom_df = pd.DataFrame(chronos_anomalies)
print(f"\nChronos anomalies detected : {len(chronos_anom_df)}")

if not chronos_anom_df.empty:
    print("\nTop 10 by z-score:")
    print(
        chronos_anom_df.sort_values("z_score", ascending=False)
        .head(10)[["commodity", "country", "year", "actual", "predicted_median", "z_score"]]
        .round(2)
        .to_string(index=False)
    )

## 3. Embedding-Based Anomaly Detection

Sentence-transformer models map text to dense semantic vectors.  We exploit this to
detect **semantically anomalous production records** — records whose natural-language
description is far from the centroid of all descriptions for the same commodity.

**Algorithm:**

1. For every row in `df_yoy`, compose a text description:
   > *"China cobalt production in 2016: 65,000 tonnes. Year-over-year change: +12.3%"*
2. Encode all texts with `BAAI/bge-large-en-v1.5` (L2-normalised).
3. For each commodity, compute the centroid of all its embeddings.
4. Flag records whose cosine distance from the centroid exceeds the **95th percentile**
   within that commodity.

This catches records that differ semantically from the commodity's "typical story" —
e.g., a country that produces an unusually large quantity with an extreme YoY swing.

In [ ]:
print("Loading sentence-transformer (BAAI/bge-large-en-v1.5) …")
emb_model = SentenceTransformer("BAAI/bge-large-en-v1.5")
print("Model loaded.")

# Build text descriptions
texts = df_yoy.apply(
    lambda r: (
        f"{r['country']} {r['commodity']} production in {int(r['year'])}: "
        f"{r['quantity']:,.0f} tonnes. "
        f"Year-over-year change: {r['yoy_change'] * 100:.1f}%"
    ),
    axis=1,
).tolist()

print(f"Encoding {len(texts):,} records …")
embeddings = emb_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True,
)

# L2-normalise so cosine similarity = dot product
norms      = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings = embeddings / (norms + 1e-10)

print(f"Embedding matrix shape : {embeddings.shape}")

In [ ]:
# Per-commodity centroid + cosine-distance anomaly detection
EMBEDDING_ANOMALY_PERCENTILE = 95  # top 5% most distant = anomalous

embedding_anomaly_parts = []

for commodity in df_yoy["commodity"].unique():
    mask           = (df_yoy["commodity"] == commodity).values
    commodity_embs = embeddings[mask]

    if len(commodity_embs) < 5:
        continue

    centroid  = commodity_embs.mean(axis=0, keepdims=True)  # (1, D)
    centroid  = centroid / (np.linalg.norm(centroid) + 1e-10)
    cos_sims  = cosine_similarity(commodity_embs, centroid).flatten()
    distances = 1.0 - cos_sims  # cosine distance

    threshold  = np.percentile(distances, EMBEDDING_ANOMALY_PERCENTILE)
    anom_mask  = distances > threshold

    commodity_df                   = df_yoy[mask].copy()
    commodity_df["emb_distance"]   = distances
    commodity_df["is_anomaly_emb"] = anom_mask

    embedding_anomaly_parts.append(commodity_df[anom_mask])

emb_anom_df = pd.concat(embedding_anomaly_parts, ignore_index=True)

print(f"Embedding anomalies : {len(emb_anom_df):,}")
print("\nTop 10 by cosine distance:")
print(
    emb_anom_df.sort_values("emb_distance", ascending=False)
    .head(10)[["commodity", "country", "year", "quantity", "yoy_change", "emb_distance"]]
    .round(4)
    .to_string(index=False)
)

## 4. PyTorch Autoencoder Anomaly Detection

An **autoencoder** learns to compress and reconstruct its inputs.  Because it is trained
to minimise reconstruction error across *all* series, typical series are reconstructed
well; unusual series — with patterns the model has not seen — incur high reconstruction
error and are flagged as anomalies.

**Feature set (per series):**

| Feature | Description |
|---|---|
| `mean_prod` | Average production quantity |
| `std_prod` | Standard deviation of quantity |
| `trend` | Linear trend slope (OLS) |
| `mean_yoy` | Mean year-over-year change |
| `std_yoy` | Std of YoY changes |
| `max_drop` | Largest single-year decline (most negative YoY) |
| `max_spike` | Largest single-year spike (most positive YoY) |
| `volatility` | Coefficient of variation (std / mean) |

**Architecture:** 8 → 16 → 4 (latent) → 16 → 8 with ReLU activations, trained for 200 epochs with Adam.

The **top 10%** by reconstruction error are flagged as anomalous.

In [ ]:
import torch.nn as nn

# ── Build series-level feature matrix ───────────────────────────────────────
feature_records = []
for (commodity, country), g in df.groupby(["commodity", "country"]):
    if len(g) < 5:
        continue
    q   = g["quantity"].values
    yoy = g["yoy_change"].dropna().values if "yoy_change" in g.columns else np.array([])

    feature_records.append({
        "commodity" : commodity,
        "country"   : country,
        "mean_prod" : np.mean(q),
        "std_prod"  : np.std(q),
        "trend"     : np.polyfit(range(len(q)), q, 1)[0] if len(q) > 1 else 0.0,
        "mean_yoy"  : np.mean(yoy)  if len(yoy) > 0 else 0.0,
        "std_yoy"   : np.std(yoy)   if len(yoy) > 0 else 0.0,
        "max_drop"  : np.min(yoy)   if len(yoy) > 0 else 0.0,
        "max_spike" : np.max(yoy)   if len(yoy) > 0 else 0.0,
        "volatility": np.std(q) / (np.mean(q) + 1e-6),
    })

features_df  = pd.DataFrame(feature_records)
feature_cols = ["mean_prod", "std_prod", "trend", "mean_yoy",
                "std_yoy", "max_drop", "max_spike", "volatility"]

scaler   = StandardScaler()
X        = scaler.fit_transform(features_df[feature_cols].fillna(0).values)
X_tensor = torch.FloatTensor(X)

print(f"Series-level feature matrix : {X.shape}")

# ── Autoencoder architecture ──────────────────────────────────────────────
class Autoencoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))

ae        = Autoencoder(len(feature_cols))
optimizer = torch.optim.Adam(ae.parameters(), lr=0.001)
criterion = nn.MSELoss(reduction="none")

# ── Train ──────────────────────────────────────────────────────────────────────
print("Training autoencoder …")
ae.train()
for epoch in range(200):
    optimizer.zero_grad()
    output = ae(X_tensor)
    loss   = criterion(output, X_tensor).mean()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch + 1:3d}/200 — loss: {loss.item():.6f}")

# ── Reconstruction error as anomaly score ─────────────────────────────────
ae.eval()
with torch.no_grad():
    recon       = ae(X_tensor)
    recon_error = criterion(recon, X_tensor).mean(dim=1).numpy()

features_df["recon_error"]   = recon_error
ae_threshold                  = np.percentile(recon_error, 90)
features_df["is_anomaly_ae"] = recon_error > ae_threshold

n_ae = features_df["is_anomaly_ae"].sum()
print(f"\nAutoencoder anomalous series : {n_ae:,} (threshold = {ae_threshold:.4f})")
print("\nTop 10 by reconstruction error:")
print(
    features_df[features_df["is_anomaly_ae"]]
    .sort_values("recon_error", ascending=False)
    .head(10)[["commodity", "country", "mean_prod", "volatility", "max_drop", "max_spike", "recon_error"]]
    .round(4)
    .to_string(index=False)
)

## 5. Method Comparison

We now synthesise results across all four methods:

* **Summary table** — method type, anomaly count, and agreement with other methods.
* **Overlap analysis** — how many (commodity, country, year) triples are flagged by
  multiple methods simultaneously.
* **Unique detections** — what Chronos / embeddings find that the classical methods miss.

In [ ]:
# ── Harmonise all anomaly sets to (commodity, country, year) triples ──────────

# 1. Z-score — already at point level
set_zscore = set(
    zip(
        df_yoy.loc[df_yoy["is_anomaly_zscore"], "commodity"],
        df_yoy.loc[df_yoy["is_anomaly_zscore"], "country"],
        df_yoy.loc[df_yoy["is_anomaly_zscore"], "year"],
    )
)

# 2. Isolation Forest — series-level; expand to all years in those series
if_pairs = set(
    zip(
        classical_feat_df.loc[classical_feat_df["is_anomaly_if"], "commodity"],
        classical_feat_df.loc[classical_feat_df["is_anomaly_if"], "country"],
    )
)
set_if = set(
    (r.commodity, r.country, r.year)
    for _, r in df_yoy.iterrows()
    if (r.commodity, r.country) in if_pairs
)

# 3. Chronos — point level
if not chronos_anom_df.empty:
    set_chronos = set(
        zip(
            chronos_anom_df["commodity"],
            chronos_anom_df["country"],
            chronos_anom_df["year"],
        )
    )
else:
    set_chronos = set()

# 4. Embeddings — point level
set_emb = set(
    zip(
        emb_anom_df["commodity"],
        emb_anom_df["country"],
        emb_anom_df["year"],
    )
)

# 5. Autoencoder — series-level; expand to all years
ae_pairs = set(
    zip(
        features_df.loc[features_df["is_anomaly_ae"], "commodity"],
        features_df.loc[features_df["is_anomaly_ae"], "country"],
    )
)
set_ae = set(
    (r.commodity, r.country, r.year)
    for _, r in df_yoy.iterrows()
    if (r.commodity, r.country) in ae_pairs
)

# ── Summary table ───────────────────────────────────────────────────────────
all_methods = {
    "Z-score"          : set_zscore,
    "Isolation Forest" : set_if,
    "Chronos"          : set_chronos,
    "Embeddings"       : set_emb,
    "Autoencoder"      : set_ae,
}

summary_rows = []
for name, s in all_methods.items():
    others         = {k: v for k, v in all_methods.items() if k != name}
    union_others   = set().union(*others.values()) if others else set()
    agreement      = len(s & union_others)
    unique_to_this = len(s - union_others)
    method_type    = "Classical" if name in ("Z-score", "Isolation Forest") else "FM-based"
    granularity    = "Point" if name in ("Z-score", "Chronos", "Embeddings") else "Series"
    summary_rows.append({
        "Method"            : name,
        "Type"              : method_type,
        "Granularity"       : granularity,
        "Anomalies Found"   : len(s),
        "Agree w/ Others"   : agreement,
        "Unique Detections" : unique_to_this,
    })

summary_df = pd.DataFrame(summary_rows)
print("Method Comparison Summary:")
print(summary_df.to_string(index=False))

In [ ]:
# ── Visualisation 1: Bar chart of anomaly counts ─────────────────────────────
fig_bar = px.bar(
    summary_df,
    x="Method",
    y="Anomalies Found",
    color="Type",
    color_discrete_map={"Classical": "steelblue", "FM-based": "crimson"},
    pattern_shape="Granularity",
    text="Anomalies Found",
    title="Anomaly Counts by Detection Method",
)
fig_bar.update_traces(textposition="outside")
fig_bar.update_layout(
    plot_bgcolor="white", paper_bgcolor="white",
    yaxis_title="# Anomalous (commodity, country, year) triples",
    height=450,
)
fig_bar.show()

# ── Visualisation 2: Overlap heatmap ────────────────────────────────────────
method_names   = list(all_methods.keys())
n_methods      = len(method_names)
overlap_matrix = np.zeros((n_methods, n_methods), dtype=int)
for i, n1 in enumerate(method_names):
    for j, n2 in enumerate(method_names):
        overlap_matrix[i, j] = len(all_methods[n1] & all_methods[n2])

# Build annotated heatmap with explicit annotation strings (avoids text_auto)
annotations = [
    dict(
        x=method_names[j],
        y=method_names[i],
        text=str(overlap_matrix[i, j]),
        showarrow=False,
        font=dict(color="white" if overlap_matrix[i, j] > overlap_matrix.max() * 0.5 else "black"),
    )
    for i in range(n_methods)
    for j in range(n_methods)
]

fig_hm = go.Figure(
    data=go.Heatmap(
        z=overlap_matrix,
        x=method_names,
        y=method_names,
        colorscale="Blues",
        colorbar=dict(title="Shared anomalies"),
    )
)
fig_hm.update_layout(
    title="Method Agreement Matrix — Shared (commodity, country, year) Anomalies",
    height=450,
    plot_bgcolor="white",
    paper_bgcolor="white",
    annotations=annotations,
)
fig_hm.show()

# ── Visualisation 3: Unique vs shared breakdown ────────────────────────────
fig_uniq = px.bar(
    summary_df,
    x="Method",
    y=["Agree w/ Others", "Unique Detections"],
    color_discrete_sequence=["#4472C4", "#ED7D31"],
    barmode="stack",
    title="Shared vs Unique Anomaly Detections by Method",
)
fig_uniq.update_layout(
    plot_bgcolor="white", paper_bgcolor="white",
    yaxis_title="# (commodity, country, year) triples",
    height=450,
    legend_title="Detection type",
)
fig_uniq.show()

In [ ]:
# ── Plot top 6 series flagged by Chronos but NOT by classical methods ─────────
classical_union = set_zscore | set_if
chronos_novel   = set_chronos - classical_union

print(f"Chronos anomalies             : {len(set_chronos)}")
print(f"Chronos novel (not classical) : {len(chronos_novel)}")

if not chronos_anom_df.empty and chronos_novel:
    novel_triples_set = set(chronos_novel)

    novel_df = chronos_anom_df[
        chronos_anom_df.apply(
            lambda r: (r["commodity"], r["country"], r["year"]) in novel_triples_set,
            axis=1,
        )
    ].copy()

    # Pick top 6 series by max z-score
    top_novel_series = (
        novel_df.groupby(["commodity", "country"])["z_score"].max()
        .sort_values(ascending=False)
        .head(6)
        .reset_index()[["commodity", "country"]]
        .values.tolist()
    )

    n_cols = 3
    n_rows = (len(top_novel_series) + n_cols - 1) // n_cols

    subplot_titles = [
        f"{commodity.title()} — {country}"
        for commodity, country in top_novel_series
    ]

    fig_novel = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=subplot_titles,
        vertical_spacing=0.16,
        horizontal_spacing=0.08,
    )

    for idx, (commodity, country) in enumerate(top_novel_series):
        row         = idx // n_cols + 1
        col         = idx %  n_cols + 1
        show_legend = (idx == 0)

        series_df = df[
            (df["commodity"] == commodity) & (df["country"] == country)
        ].sort_values("year")

        c_anom = novel_df[
            (novel_df["commodity"] == commodity) & (novel_df["country"] == country)
        ]

        fig_novel.add_trace(
            go.Scatter(
                x=series_df["year"], y=series_df["quantity"],
                mode="lines+markers",
                line=dict(color="steelblue", width=2),
                marker=dict(size=4, color="steelblue"),
                name="Production",
                showlegend=show_legend,
                hovertemplate="%{x}: %{y:,.0f}<extra></extra>",
            ),
            row=row, col=col,
        )

        if len(c_anom) > 0:
            anom_qty = pd.merge(
                c_anom[["year"]],
                series_df[["year", "quantity"]],
                on="year",
                how="left",
            )
            fig_novel.add_trace(
                go.Scatter(
                    x=anom_qty["year"], y=anom_qty["quantity"],
                    mode="markers",
                    marker=dict(symbol="x", size=12, color="darkorange",
                                line=dict(width=2)),
                    name="Chronos novel anomaly",
                    showlegend=show_legend,
                    hovertemplate="Chronos anomaly %{x}: %{y:,.0f}<extra></extra>",
                ),
                row=row, col=col,
            )

    fig_novel.update_layout(
        title_text="Chronos Novel Anomalies — Flagged by Chronos but NOT by Classical Methods",
        height=520 * n_rows,
        plot_bgcolor="white",
        paper_bgcolor="white",
    )
    fig_novel.update_xaxes(gridcolor="#eeeeee")
    fig_novel.update_yaxes(gridcolor="#eeeeee")
    fig_novel.show()

else:
    print("No Chronos-novel anomalies to plot.")
    print("Re-run with a broader COMMODITIES_TO_CHECK or a lower CHRONOS_Z_THRESHOLD.")

## Summary — Classical vs Foundation Model Anomaly Detection

### What each method detects

| Method | What it finds | Strengths | Limitations |
|---|---|---|---|
| **Z-score** | YoY changes extreme relative to that series' own history | Fast, interpretable, no ML needed | Assumes Gaussian YoY distribution; misses structural breaks and slow drifts |
| **Isolation Forest** | Series whose statistical fingerprint is hard to isolate | Handles multivariate feature space; no distributional assumption | Hand-crafted features; series-level only — loses temporal resolution |
| **Chronos** | Years where the FM's probabilistic forecast is wrong | Leverages transfer learning from millions of real time-series; adapts to each series' history dynamically | Computationally expensive; requires ≥ 5 years of context; small quantile model may under-represent tails |
| **Embeddings** | Records whose semantic description is far from the commodity norm | Captures natural-language semantics; robust to data scale differences | Anomaly threshold is relative (top 5%); may flag valid extremes if a commodity is inherently volatile |
| **Autoencoder** | Series whose joint feature pattern the model cannot reconstruct | Learns non-linear feature interactions end-to-end; no manual feature engineering needed | Series-level only; requires training (though fast here); sensitive to feature scaling |

### Key findings

1. **Agreement is the strongest signal.** Points flagged by multiple independent methods — especially Chronos (which has no knowledge of classical features) and Z-score — are the most credible anomalies to investigate first.

2. **Chronos adds temporal context.** Classical Z-score is blind to whether a large YoY swing is expected given recent trend.  Chronos conditions on all prior years, so a high-growth series that suddenly plateaus may not look anomalous by Z-score (the plateau is statistically small relative to the spike that preceded it) but *is* anomalous to Chronos because it breaks the trend.

3. **Embedding anomalies reflect rarity, not just magnitude.** A country that uniquely combined very small production with a large YoY swing may be semantically far from its commodity's centroid even if neither value alone is extreme — embeddings catch these joint outliers.

4. **Autoencoder complements Isolation Forest.** Both operate on series-level features, but the autoencoder's non-linear encoder can capture interactions (e.g., high volatility *and* high mean simultaneously) that the linear Isolation Forest feature space may miss.

### Recommended workflow

1. **Tier-1 priority:** (commodity, country, year) triples flagged by ≥ 3 methods — highest confidence anomalies.
2. **Tier-2:** Chronos-only or embedding-only anomalies — investigate as potential novel detections.
3. **Validation:** Cross-reference `table_notes` / `figure_notes` columns; data-quality artefacts (revisions, reclassifications) vs genuine supply disruptions require domain context.

### Next steps

* Scale Chronos to all commodities (currently scoped to a subset for speed).
* Experiment with larger Chronos variants (`chronos-t5-base`, `chronos-t5-large`) for improved forecast calibration.
* Enrich embedding texts with geopolitical context (sanctions, conflict) for richer semantic separation.
* Feed multi-method agreement scores into `08_geopolitical_scenario_modeling.ipynb` as a disruption-risk feature.